# Pareja C — Cierre: del registro al modelo final

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/11909/Poster2Movie-Clasificador/blob/main/notebooks/C_cierre.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/11909/Poster2Movie-Clasificador/blob/main/notebooks/C_cierre_kaggle.ipynb)

Junta lo que funcionó de todas las ideas, arma el modelo final y lo deja listo para el release. **Solo
validación**: test se mide una sola vez, en `C_release`.

Estado al 27 nov (validación, base: F1 macro 0.544, promedio de los 4 débiles 0.384):

| Experimento | F1 macro | Débiles | |
| --- | --- | --- | --- |
| `A7_l14` (CLIP L/14) | 0.605 (+0.061) | 0.429 (+0.045) | el salto grande |
| `A1_3recortes` (B/32) | 0.563 (+0.019) | 0.391 (+0.008) | ayuda: probarlo en L/14 |
| `A7_siglip` | 0.560 (+0.016) | 0.379 (−0.005) | diversidad para el ensamble |
| `B_combinado_sinpw` (B/32, 4 semillas) | 0.544 a 0.555 | 0.374 a 0.399 | mejora dentro del ruido: probarla en L/14 |
| `A2_ocr` | — | — | solo 195 películas de val (muestra): no comparable, no entra al ensamble |

Orden: (1) vectores L/14 con 3 recortes (GPU) → (2) experimentos nuevos sobre L/14 → (3) miembros →
(4) ensambles → (5) apilamiento → (6) tabla final → (7) memoria en T4 → (8) release con `C_release`.
Si no hay GPU, salten el paso 1: el resto corre en CPU y el experimento que falte se salta solo.

## Preparar el entorno

**GPU (T4)** para los pasos 1 y 7; el resto basta con CPU.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`C-cierre`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "C-cierre"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt
!nvidia-smi -L

## 1. Vectores de CLIP L/14 con 3 recortes (GPU, opcional)

Tres pasadas de L/14 por cartel: lo más caro del cierre. Guarda por partes; si Colab se desconecta, al
volver a correrlo continúa. Sin GPU: `A7_codificadores_kaggle.ipynb` con `clip_l14 --variante 3recortes`.

In [ ]:
!python -m clasificador.codificadores clip_l14 --variante 3recortes

## 2. Experimentos nuevos sobre L/14 (CPU)

- `A1_l14_3recortes`: idea 1 sobre el mejor codificador.
- `B_l14_combinado_sinpw` (+ `_s1`, `_s2`): la mejor receta de B (MLP + focal) sobre L/14, tres semillas.

El que no tenga sus vectores se salta (p. ej. si no corrieron el paso 1).

In [ ]:
from clasificador import rutas
from clasificador.entrenar import cargar_experimento

NUEVOS = ["A1_l14_3recortes", "B_l14_combinado_sinpw", "B_l14_combinado_sinpw_s1", "B_l14_combinado_sinpw_s2"]
for nombre in NUEVOS:
    try:
        for v in cargar_experimento(f"experimentos/{nombre}.yaml")["vectores"]:
            rutas.ruta_vectores(v)
    except FileNotFoundError as e:
        print(f"Salto {nombre}: {e}")
        continue
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}
!python -m clasificador.ranking A7_
!python -m clasificador.ranking A1_
!python -m clasificador.ranking B_l14

## 3. Miembros: el mejor experimento de cada codificador

`clasificador/cierre.py`: por codificador, el primero según `ranking` (cumple la regla > débiles > F1
macro). Deja fuera los combinados y los evaluados con menos películas que la base (OCR).

In [ ]:
from clasificador.cierre import mejores_por_codificador, COLUMNAS
mejores = mejores_por_codificador()
MIEMBROS = list(mejores.index)
print("MIEMBROS =", MIEMBROS)
mejores[COLUMNAS]

## 4. Ensambles (idea 7b)

- `C7_top2`: los dos mejores (por F1 macro).
- `C7_todos`: uno por codificador.
- `C7_todos_3recortes`: además `A1_3recortes` (B/32 viendo el cartel completo), si no está ya.

In [ ]:
ENSAMBLES = {"C7_top2": MIEMBROS[:2], "C7_todos": MIEMBROS}
if "A1_3recortes" not in MIEMBROS:
    ENSAMBLES["C7_todos_3recortes"] = MIEMBROS + ["A1_3recortes"]
for nombre, miembros in ENSAMBLES.items():
    if len(miembros) < 2:
        continue
    !python -m clasificador.ensamble {nombre} {" ".join(miembros)}
    !python -m clasificador.evaluar {nombre}

## 5. Apilamiento (idea 6)

Etapa 2 lineal residual con predicciones fuera de pliegue (5 pliegues por franquicia). Reentrena cada
miembro 5 veces: con L/14 y cabezas MLP tarda unos minutos por miembro (CPU basta).

- `C6_mejor`: sobre el mejor modelo solo.
- `C6_todos`: sobre los miembros del ensamble (apilar el ensamble).

Cada uno deja `resultados/<nombre>/pesos_etapa2.csv` y `mapa_calor.png` para el reporte.

In [ ]:
APILADOS = {"C6_mejor": MIEMBROS[:1], "C6_todos": MIEMBROS}
for nombre, miembros in APILADOS.items():
    !python -m clasificador.apilamiento {nombre} {" ".join(miembros)}
    !python -m clasificador.evaluar {nombre}

## 6. Tabla final (validación)

Los miembros y todos los combinados contra la base. El candidato es el primero: cumple la regla
(macro no baja y los débiles suben) y tiene el mejor promedio de los débiles. Si dos están a menos de
~0.005 (el ruido entre semillas), elijan el más ligero (menos codificadores en la demo).

In [ ]:
from clasificador.cierre import tabla_completa
tabla = tabla_completa()
final = tabla[tabla.index.isin(MIEMBROS) | tabla.index.str.startswith("C")]
final = final.sort_values(["cumple", "debiles_prom", "F1_macro"], ascending=False)
final[COLUMNAS]

In [ ]:
from IPython.display import Image, display
CANDIDATO = final.index[0]
print("Candidato:", CANDIDATO)
!python -m clasificador.comparar base {CANDIDATO}
!python -m clasificador.comparar {MIEMBROS[0]} {CANDIDATO}
for nombre in APILADOS:
    if os.path.exists(f"resultados/{nombre}/mapa_calor.png"):
        display(Image(f"resultados/{nombre}/mapa_calor.png", width=500))

## 7. ¿Cabe en la demo? (GPU)

La demo corre en una T4 junto con la reseña (Qwen3-VL-2B) y el cartel (SD-Turbo). Si el candidato
no cabe, elijan el mejor de la tabla con menos codificadores.

In [ ]:
import torch
from PIL import Image as PILImage
from clasificador.predictor import Clasificador
for nombre in final.index[:3]:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    clasificador = Clasificador(rutas.ruta_modelo(nombre))
    clasificador.predecir(PILImage.new("RGB", (384, 576)))
    print(f"{nombre:<24} pico de memoria GPU: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    del clasificador

## 8. Release

Con el candidato elegido, abran **`C_release`** con `TAG = "v0.2.0"` y `EXPERIMENTO = "<candidato>"`:
mide val **y test** (la única vez que se mira test) y arma `releases/v0.2.0/`. Antes suban esta rama
(celda de abajo) para que el registro y los mapas de calor queden en el repo.

Luego, para la demo: PR al repo principal con `RELEASE_CLASIFICADOR`, y copiar a `p2m/codificadores.py`
los codificadores y variantes que use el candidato (L/14, SigLIP, 3 recortes) y a `construir_cabeza`
las cabezas nuevas (GELU, LayerNorm). Ver `docs/PAREJA_C.md`.

## Subir a la rama `C-cierre`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`C-cierre`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Personal access tokens → Tokens (classic)*, con el permiso `repo` (los *fine-grained* no sirven: el repo es de otra cuenta personal)).

In [ ]:
from getpass import getpass
import base64, os, subprocess


def empujar(token, destino):
    # Sube a GitHub SIN poner el token en la URL (no aparece en mensajes ni salidas).
    # Si el token no sirve, da un error claro en vez de quedarse esperando una contraseña.
    token = (token or "").strip()
    if token.startswith("github_pat_"):
        raise SystemExit("Ese es un token fine-grained (github_pat_...): con este repo no sirve. "
                         "Crea uno classic (ghp_...) con el permiso repo.")
    cabecera = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader={cabecera}", "push", "-q",
                        f"https://github.com/{REPO}.git", destino],
                       capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        raise SystemExit("No se pudo subir a GitHub: " + r.stderr.replace(token, "***").strip()[-400:] +
                         "\nRevisa que el token sea classic con permiso repo y que tu cuenta sea colaboradora del repo.")
    print("Subido a GitHub:", destino)

!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
empujar(token, f"HEAD:{RAMA}")
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")